# CIV Batch PyQSOFit Workflow

This copy reads `Target_lists/target-data.csv`, builds each target's FITS path and fitting parameters, runs the four CIV model configurations, and writes one combined CSV of fit outputs.

In [1]:
%matplotlib inline
from pathlib import Path
import ast
import re
import tempfile
import timeit
import warnings

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from astropy.io import fits
from astropy.table import Table

import sys
sys.path.append('../')
from pyqsofit.PyQSOFit import QSOFit
from data_calculations import blueshift_calc, asym_calc

warnings.filterwarnings('ignore')
QSOFit.set_mpl_style()

In [2]:
# Input/output configuration
PROJECT_ROOT = Path('/Users/nayera/PyQSOFit')
TARGET_CSV = PROJECT_ROOT / 'Target_lists' / 'target-data.csv'
SPECTRA_ROOT = PROJECT_ROOT / 'CIV_galaxy_spectra_targets'
TRACKING_CSV = PROJECT_ROOT / 'Target_lists' / 'civ_batch_fit_inputs_1600.csv'
RESULTS_CSV = PROJECT_ROOT / 'Target_lists' / 'civ_batch_fit_outputs_1600.csv'

# MCMC is intentionally run only once per target/model fit.
RUN_MCMC = True
NSAMP = 200
NBURN = 20
NTHIN = 10

# The old notebook down-weighted low-S/N pixels rather than dropping them.
SNR_THRESHOLD = 3.0
LOW_SNR_ERROR_SCALE = 1e6
MAX_CIV_FWHM_KMS_SIGMA = 0.02157
NARROW_FWHM_LIMIT_KMS = 1600
NARROW_CIV_MAXSIG = 0.0022664260627611094
BROAD_FWHM_THRESHOLD = NARROW_FWHM_LIMIT_KMS
CIV_BROAD_PROFILE_GRID_POINTS = 5000
CIV_BROAD_PROFILE_NSIGMA = 6.0

MODEL_GRID = {
    'A': {'br_component': 3, 'na_component': 0},
    'B': {'br_component': 2, 'na_component': 1},
    'C': {'br_component': 2, 'na_component': 0},
    'D': {'br_component': 1, 'na_component': 1},
}

# Set to a small integer for a smoke test, then back to None for the full run.
TARGET_LIMIT = None


In [3]:
def _numbers_from_cell(value):
    if pd.isna(value):
        return []
    text = str(value).strip()
    if not text or text.lower() in {'nan', 'none', 'none needed', 'no mask', 'no masks'}:
        return []
    return [float(x) for x in re.findall(r'[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?', text)]


def parse_range(value):
    nums = _numbers_from_cell(value)
    if len(nums) != 2:
        return None
    lo, hi = sorted(nums)
    return np.array([lo, hi], dtype=float)


def parse_mask(value):
    nums = _numbers_from_cell(value)
    if not nums:
        return None
    if len(nums) % 2 != 0:
        return None
    pairs = np.array(nums, dtype=float).reshape(-1, 2)
    pairs.sort(axis=1)
    return pairs


def as_serializable(value):
    if value is None:
        return ''
    arr = np.asarray(value)
    return arr.tolist()


def load_targets(target_csv=TARGET_CSV):
    df = pd.read_csv(target_csv)
    df = df[df['target name'].notna() & df['file name'].notna()].copy()
    df['target_name'] = df['target name'].astype(str).str.strip()
    df['file_name'] = df['file name'].astype(str).str.strip()
    df['fits_path'] = df.apply(lambda row: SPECTRA_ROOT / row['target_name'] / row['file_name'], axis=1)
    df['z_o3_numeric'] = pd.to_numeric(df['z (o3)'], errors='coerce')
    df['wu_z_numeric'] = pd.to_numeric(df['Wu_z'], errors='coerce')
    use_wu_z = (df['target_name'] == 'J1534+5711') & df['z_o3_numeric'].isna() & df['wu_z_numeric'].notna()
    df['fit_z'] = df['z_o3_numeric'].where(~use_wu_z, df['wu_z_numeric'])
    df['fit_z_source'] = np.where(use_wu_z, 'Wu_z', 'z (o3)')
    df.loc[df['fit_z'].isna(), 'fit_z_source'] = ''
    df['range_arr'] = df['continuum fit range'].apply(parse_range)
    df['mask_arr'] = df['masked sections'].apply(parse_mask)
    df['fits_exists'] = df['fits_path'].apply(lambda p: p.exists())
    df['valid_fit_z'] = df['fit_z'].notna()
    df['valid_range'] = df['range_arr'].apply(lambda x: x is not None)
    df['run_fit'] = df['fits_exists'] & df['valid_fit_z'] & df['valid_range']

    reasons = []
    for row in df.itertuples(index=False):
        missing = []
        if not row.fits_exists:
            missing.append('missing FITS file')
        if not row.valid_fit_z:
            missing.append('missing/non-numeric fit redshift')
        if not row.valid_range:
            missing.append('missing/invalid continuum fit range')
        reasons.append('; '.join(missing))
    df['skip_reason'] = reasons

    export = df.copy()
    export['fits_path'] = export['fits_path'].astype(str)
    export['range_arr'] = export['range_arr'].apply(as_serializable)
    export['mask_arr'] = export['mask_arr'].apply(as_serializable)
    export.to_csv(TRACKING_CSV, index=False)
    return df


targets = load_targets()
fit_targets = targets[targets['run_fit']].copy()
if TARGET_LIMIT is not None:
    fit_targets = fit_targets.head(TARGET_LIMIT).copy()

print(f'Loaded {len(targets)} rows with FITS filenames from {TARGET_CSV.name}.')
print(f'{len(fit_targets)} targets are ready to fit. Tracking table written to {TRACKING_CSV}.')
if (~targets['run_fit']).any():
    display(targets.loc[~targets['run_fit'], ['target_name', 'file_name', 'z (o3)', 'Wu_z', 'skip_reason']])

display(fit_targets[['target_name', 'file_name', 'fit_z', 'fit_z_source', 'range_arr', 'mask_arr', 'fits_path']])

Loaded 14 rows with FITS filenames from target-data.csv.
14 targets are ready to fit. Tracking table written to /Users/nayera/PyQSOFit/Target_lists/civ_batch_fit_inputs_1600.csv.


,target_name,file_name,fit_z,fit_z_source,range_arr,mask_arr,fits_path
0,J1245+4348,01spec-6617-56365-0860.fits,4.890,z (o3),"[1445.0, 1600.0]","[[1495.0, 1503.0]]",/Users/nayera/PyQSOFit/CIV_galaxy_spectra_targ...
1,J1100+5800,02spec-8184-57426-0330.fits,4.759,z (o3),"[1445.0, 1615.0]","[[1502.0, 1513.0], [1533.0, 1537.0]]",/Users/nayera/PyQSOFit/CIV_galaxy_spectra_targ...
2,J1328+4445,03spec-6627-56369-0578.fits,4.820,z (o3),"[1445.0, 1705.0]","[[1499.0, 1510.0], [1526.0, 1528.0], [1565.0, ...",/Users/nayera/PyQSOFit/CIV_galaxy_spectra_targ...
3,J0831+4046,04spec-3808-55513-0180.fits,4.900,z (o3),"[1445.0, 1660.0]","[[1511.0, 1514.0], [1528.0, 1531.0]]",/Users/nayera/PyQSOFit/CIV_galaxy_spectra_targ...
4,J1102+6635,05spec-7111-56741-0438.fits,4.787,z (o3),"[1400.0, 1600.0]","[[1534.0, 1544.0], [1605.0, 1650.0]]",/Users/nayera/PyQSOFit/CIV_galaxy_spectra_targ...
10,J1342+5838,11spec-6810-56447-0055.fits,4.850,z (o3),"[1400.0, 1600.0]",None,/Users/nayera/PyQSOFit/CIV_galaxy_spectra_targ...
11,J1534+5711,12spec-0615-52347-0094.fits,4.820,Wu_z,"[1400.0, 1580.0]","[[1500.0, 1503.0], [1535.0, 1538.0]]",/Users/nayera/PyQSOFit/CIV_galaxy_spectra_targ...
16,J0941+5947,17spec-8194-57401-0548.fits,4.840,z (o3),"[1445.0, 1675.0]","[[1504.0, 1512.0], [1544.0, 1549.0]]",/Users/nayera/PyQSOFit/CIV_galaxy_spectra_targ...
19,J0759+1800,20spec-4490-55629-0074.fits,4.796,z (o3),"[1425.0, 1600.0]","[[1510.0, 1525.0], [1528.0, 1533.0]]",/Users/nayera/PyQSOFit/CIV_galaxy_spectra_targ...
20,J0807+1328,21spec-4505-55603-0810.fits,4.880,z (o3),"[1450.0, 1650.0]",None,/Users/nayera/PyQSOFit/CIV_galaxy_spectra_targ...


In [4]:
def continuum_windows_from_range(range_arr):
    lo, hi = map(float, range_arr)
    low_start = max(lo, 1445.0)
    low_end = min(low_start + 20.0, hi)
    high_end = hi
    high_start = max(hi - 15.0, low_end + 5.0)
    if low_end - low_start < 5.0 or high_end - high_start < 5.0 or high_start <= low_end:
        width = hi - lo
        low_start, low_end = lo, lo + 0.15 * width
        high_start, high_end = hi - 0.15 * width, hi
    return np.rec.array(
        [(low_start, low_end), (high_start, high_end)],
        formats='float32,float32',
        names='min,max',
    )


def make_qsopar(param_dir, range_arr, br_component, na_component):
    primary_hdu = fits.PrimaryHDU(header=fits.Header({'Author': 'Nayera Abdessalam'}))

    line_priors = np.rec.array([
        (1549.06, 'CIV', 1450, 1700, 'CIV_br', br_component, 0.0, 0.0, 1e10,
         0.005, 0.004, MAX_CIV_FWHM_KMS_SIGMA, 0.015, 0, 0, 0, 0.05, 1),
        (1549.06, 'CIV', 1450, 1700, 'CIV_na', na_component, 0.0, 0.0, 1e10,
         1e-3, 5e-4, NARROW_CIV_MAXSIG, 0.01, 1, 1, 0, 0.002, 1),
    ], formats='float32,a20,float32,float32,a20,int32,float32,float32,float32,float32,float32,float32,float32,int32,int32,int32,float32,int32',
       names='lambda,compname,minwav,maxwav,linename,ngauss,inisca,minsca,maxsca,inisig,minsig,maxsig,voff,vindex,windex,findex,fvalue,vary')
    hdu1 = fits.BinTableHDU(data=line_priors, name='line_priors')

    hdu2 = fits.BinTableHDU(data=continuum_windows_from_range(range_arr), name='conti_windows')

    conti_priors = np.rec.array([
        ('Fe_uv_norm', 0.0, 0.0, 1e10, 1),
        ('Fe_uv_FWHM', 3000, 1200, 18000, 1),
        ('Fe_uv_shift', 0.0, -0.01, 0.01, 1),
        ('Fe_op_norm', 0.0, 0.0, 1e10, 1),
        ('Fe_op_FWHM', 3000, 1200, 18000, 1),
        ('Fe_op_shift', 0.0, -0.01, 0.01, 1),
        ('PL_norm', 1.0, 0.0, 1e10, 1),
        ('PL_slope', -1.5, -5.0, 3.0, 1),
        ('Blamer_norm', 0.0, 0.0, 1e10, 1),
        ('Balmer_Te', 15000, 10000, 50000, 1),
        ('Balmer_Tau', 0.5, 0.1, 2.0, 1),
        ('conti_a_0', 0.0, None, None, 1),
        ('conti_a_1', 0.0, None, None, 1),
        ('conti_a_2', 0.0, None, None, 1),
    ], formats='a20,float32,float32,float32,int32', names='parname,initial,min,max,vary')
    hdu3 = fits.BinTableHDU(data=conti_priors, name='conti_priors')

    measure_info = Table([[[1350, 1450, 3000, 4200, 5100]], [[[1500, 1550]]]],
                         names=['cont_loc', 'Fe_flux_range'], dtype=['float32', 'float32'])
    hdu4 = fits.BinTableHDU(data=measure_info, name='measure_info')

    param_path = Path(param_dir) / 'qsopar.fits'
    fits.HDUList([primary_hdu, hdu1, hdu2, hdu3, hdu4]).writeto(param_path, overwrite=True)
    return param_path


In [5]:
def read_sdss_spectrum(fits_path):
    with fits.open(fits_path) as data:
        lam = 10 ** data[1].data['loglam']
        flux = np.asarray(data[1].data['flux'], dtype=float)
        ivar = np.asarray(data[1].data['ivar'], dtype=float)
        err = np.full_like(flux, np.inf, dtype=float)
        good_ivar = np.isfinite(ivar) & (ivar > 0)
        err[good_ivar] = 1 / np.sqrt(ivar[good_ivar])

        header = data[0].header
        meta = {
            'ra': header.get('plug_ra', np.nan),
            'dec': header.get('plug_dec', np.nan),
            'plateid': header.get('plateid', -1),
            'mjd': header.get('mjd', -1),
            'fiberid': header.get('fiberid', -1),
        }

    good = np.isfinite(flux) & np.isfinite(err) & (err > 0)
    snr = np.zeros_like(flux, dtype=float)
    snr[good] = np.abs(flux[good]) / err[good]
    low_snr = snr < SNR_THRESHOLD
    err[low_snr] *= LOW_SNR_ERROR_SCALE
    meta['snr_masked_pixels'] = int(np.sum(low_snr))
    meta['total_pixels'] = int(len(flux))
    meta['snr_masked_fraction'] = float(np.sum(low_snr) / len(flux))
    return lam, flux, err, meta


def get_line_value(q, key, default=np.nan):
    idx = np.where(np.asarray(q.line_result_name) == key)[0]
    if idx.size == 0:
        return default
    return float(np.ravel(q.line_result[idx[0]])[0])


def component_props(q, line_name, line_type, ln_sigma_br=NARROW_CIV_MAXSIG):
    names = ['FWHM', 'sigma', 'EW', 'peak', 'area', 'snr']
    values = np.array(q.line_prop_from_name(line_name, line_type, ln_sigma_br=ln_sigma_br), dtype=float)
    out = dict(zip(names, values))

    if getattr(q, 'MCMC', False) and len(getattr(q, 'gauss_result_all', [])) > 0:
        samples = []
        for i in range(q.gauss_result_all.shape[0]):
            samples.append(q.line_prop_from_name(line_name, line_type, sample_index=i, ln_sigma_br=ln_sigma_br))
        samples = np.asarray(samples, dtype=float)
        for j, name in enumerate(names):
            finite = samples[:, j][np.isfinite(samples[:, j])]
            if finite.size:
                lo, hi = np.nanpercentile(finite, [16, 84])
                out[f'{name}_err_mcmc'] = float((hi - lo) / 2)
            else:
                out[f'{name}_err_mcmc'] = np.nan
    else:
        for name in names:
            out[f'{name}_err_mcmc'] = np.nan
    return out


def gaussian_params(q, line_name, sample_index=None):
    result_names = np.asarray(q.gauss_result_name)
    result = np.asarray(q.gauss_result, dtype=object)
    params = []
    i = 1
    while True:
        keys = [f'{line_name}_{i}_scale', f'{line_name}_{i}_centerwave', f'{line_name}_{i}_sigma']
        if not all(key in result_names for key in keys):
            break
        if sample_index is None:
            vals = [float(np.ravel(result[result_names == key][0])[0]) for key in keys]
        else:
            # gauss_result_name interleaves value/error, while gauss_result_all stores values only.
            indices = [int(np.flatnonzero(result_names == key)[0] // 2) for key in keys]
            vals = [float(q.gauss_result_all[sample_index, index]) for index in indices]
        params.append(vals)
        i += 1
    return np.asarray(params, dtype=float)


def broad_half_wavelengths(q, sample_index=None):
    params = gaussian_params(q, 'CIV_br', sample_index=sample_index)
    if params.size == 0:
        return np.nan, np.nan, np.nan
    lam = np.asarray(q.wave, dtype=float)
    sel = (lam >= 1450.0) & (lam <= 1700.0)
    if np.sum(sel) < 3:
        sel = np.ones_like(lam, dtype=bool)
    lam_w = lam[sel]
    lnlam = np.log(lam_w)
    model = np.zeros_like(lam_w, dtype=float)
    for amp, center, sigma in params:
        model += amp * np.exp(-0.5 * ((lnlam - center) / sigma) ** 2)
    if not np.any(np.isfinite(model)) or np.nanmax(model) <= 0:
        return np.nan, np.nan, np.nan

    peak = np.nanmax(model)
    i_peak = int(np.nanargmax(model))
    half_peak = 0.5 * peak

    def crossing_left():
        below = np.where(model[:i_peak] < half_peak)[0]
        if below.size == 0:
            return float(lam_w[0])
        j0 = below[-1]
        j1 = j0 + 1
        return float(np.interp(half_peak, [model[j0], model[j1]], [lam_w[j0], lam_w[j1]]))

    def crossing_right():
        below = np.where(model[i_peak + 1:] < half_peak)[0]
        if below.size == 0:
            return float(lam_w[-1])
        j1 = i_peak + 1 + below[0]
        j0 = j1 - 1
        return float(np.interp(half_peak, [model[j0], model[j1]], [lam_w[j0], lam_w[j1]]))

    lam_blue = crossing_left()
    lam_red = crossing_right()

    increments = 0.5 * (model[1:] + model[:-1]) * np.diff(lam_w)
    cum = np.cumsum(increments)
    if cum.size == 0 or cum[-1] <= 0:
        lam_half = np.nan
    else:
        lam_half = float(np.interp(0.5 * cum[-1], cum, lam_w[1:]))
    return lam_blue, lam_red, lam_half


BROAD_ASYMMETRY_INPUT_KEYS = (
    'lam_p05', 'lam_p50', 'lam_p95',
    'lam_blue_15pct', 'lam_red_15pct',
    'lam_blue_25pct', 'lam_red_25pct',
    'lam_blue_80pct', 'lam_red_80pct',
    'lam_peak',
)


def broad_asymmetry_input_wavelengths(q, sample_index=None):
    empty = {key: np.nan for key in BROAD_ASYMMETRY_INPUT_KEYS}
    params = gaussian_params(q, 'CIV_br', sample_index=sample_index)
    if params.size == 0:
        return empty

    lnlam = np.linspace(
        np.min(params[:, 1] - CIV_BROAD_PROFILE_NSIGMA * params[:, 2]),
        np.max(params[:, 1] + CIV_BROAD_PROFILE_NSIGMA * params[:, 2]),
        CIV_BROAD_PROFILE_GRID_POINTS,
    )
    lam = np.exp(lnlam)
    model = np.zeros_like(lam, dtype=float)
    for amp, center, sigma in params:
        model += amp * np.exp(-0.5 * ((lnlam - center) / sigma) ** 2)
    if not np.any(np.isfinite(model)) or np.nanmax(model) <= 0:
        return empty

    i_peak = int(np.nanargmax(model))
    peak_flux = float(model[i_peak])

    def interpolate_crossing(j0, j1, target):
        y0, y1 = model[j0], model[j1]
        if y1 == y0:
            return np.nan
        return float(lam[j0] + (target - y0) * (lam[j1] - lam[j0]) / (y1 - y0))

    def crossings(fraction):
        target = fraction * peak_flux
        left_below = np.where(model[:i_peak] < target)[0]
        right_below = np.where(model[i_peak + 1:] < target)[0]
        if left_below.size == 0 or right_below.size == 0:
            return np.nan, np.nan
        left0 = int(left_below[-1])
        right1 = int(i_peak + 1 + right_below[0])
        return (
            interpolate_crossing(left0, left0 + 1, target),
            interpolate_crossing(right1 - 1, right1, target),
        )

    lam_blue_15pct, lam_red_15pct = crossings(0.15)
    lam_blue_25pct, lam_red_25pct = crossings(0.25)
    lam_blue_80pct, lam_red_80pct = crossings(0.80)

    increments = 0.5 * (model[1:] + model[:-1]) * np.diff(lam)
    cumulative = np.concatenate(([0.0], np.cumsum(increments)))
    if cumulative[-1] <= 0:
        return empty
    lam_p05, lam_p50, lam_p95 = np.interp(
        np.array([0.05, 0.50, 0.95]) * cumulative[-1], cumulative, lam
    )

    return {
        'lam_p05': float(lam_p05),
        'lam_p50': float(lam_p50),
        'lam_p95': float(lam_p95),
        'lam_blue_15pct': lam_blue_15pct,
        'lam_red_15pct': lam_red_15pct,
        'lam_blue_25pct': lam_blue_25pct,
        'lam_red_25pct': lam_red_25pct,
        'lam_blue_80pct': lam_blue_80pct,
        'lam_red_80pct': lam_red_80pct,
        'lam_peak': float(lam[i_peak]),
    }


def posterior_error(values):
    finite = np.asarray(values, dtype=float)
    finite = finite[np.isfinite(finite)]
    if finite.size == 0:
        return np.nan
    lo, hi = np.nanpercentile(finite, [16, 84])
    return float((hi - lo) / 2)


def broad_profile_mcmc_results(q):
    samples = getattr(q, 'gauss_result_all', None)
    if not getattr(q, 'MCMC', False) or samples is None or len(samples) == 0:
        return {
            'CIV_blueshift_err_mcmc': np.nan,
            'CIV_asym_err_mcmc': np.nan,
            'lam_blue_err_mcmc': np.nan,
            'lam_red_err_mcmc': np.nan,
            'lam_half_err_mcmc': np.nan,
            **{f'{key}_err_mcmc': np.nan for key in BROAD_ASYMMETRY_INPUT_KEYS},
            'mcmc_samples_returned': 0,
            'mcmc_valid_blueshift_samples': 0,
            'mcmc_valid_asym_samples': 0,
        }

    n_samples = samples.shape[0]
    profiles = [broad_half_wavelengths(q, sample_index=i) for i in range(n_samples)]
    lam_blue = np.asarray([profile[0] for profile in profiles], dtype=float)
    lam_red = np.asarray([profile[1] for profile in profiles], dtype=float)
    lam_half = np.asarray([profile[2] for profile in profiles], dtype=float)
    asymmetry_inputs = [
        broad_asymmetry_input_wavelengths(q, sample_index=i) for i in range(n_samples)
    ]
    asymmetry_input_samples = {
        key: np.asarray([values[key] for values in asymmetry_inputs], dtype=float)
        for key in BROAD_ASYMMETRY_INPUT_KEYS
    }

    # Use PyQSOFit's continuous-model broad-line peak for each posterior sample.
    broad_properties = np.asarray([
        q.line_prop_from_name(
            'CIV_br', 'broad', sample_index=i, ln_sigma_br=NARROW_CIV_MAXSIG
        )
        for i in range(n_samples)
    ], dtype=float)
    br_peak = broad_properties[:, 3]

    blueshift = np.asarray([blueshift_calc(value) for value in lam_half], dtype=float)
    # asym_calc is imported from data_calculations.py, the authoritative equation.
    asymmetry = np.asarray([
        asym_calc(blue, red, peak)
        for blue, red, peak in zip(lam_blue, lam_red, br_peak)
    ], dtype=float)

    return {
        'CIV_blueshift_err_mcmc': posterior_error(blueshift),
        'CIV_asym_err_mcmc': posterior_error(asymmetry),
        'lam_blue_err_mcmc': posterior_error(lam_blue),
        'lam_red_err_mcmc': posterior_error(lam_red),
        'lam_half_err_mcmc': posterior_error(lam_half),
        **{
            f'{key}_err_mcmc': posterior_error(values)
            for key, values in asymmetry_input_samples.items()
        },
        'mcmc_samples_returned': int(n_samples),
        'mcmc_valid_blueshift_samples': int(np.isfinite(blueshift).sum()),
        'mcmc_valid_asym_samples': int(np.isfinite(asymmetry).sum()),
    }


In [ ]:
def fit_target_model(target_row, model_name, model_cfg):
    target = target_row['target_name']
    fits_path = Path(target_row['fits_path'])
    z = float(target_row['fit_z'])
    range_arr = np.asarray(target_row['range_arr'], dtype=float)
    mask_arr = target_row['mask_arr']
    if mask_arr is not None:
        mask_arr = np.asarray(mask_arr, dtype=float)

    lam, flux, err, meta = read_sdss_spectrum(fits_path)

    with tempfile.TemporaryDirectory(prefix='pyqsofit_civ_') as param_dir:
        make_qsopar(param_dir, range_arr, model_cfg['br_component'], model_cfg['na_component'])
        q = QSOFit(lam, flux, err, z,
                   ra=meta['ra'], dec=meta['dec'], plateid=meta['plateid'],
                   mjd=meta['mjd'], fiberid=meta['fiberid'], path=param_dir)

        start = timeit.default_timer()
        q.Fit(name=f'{target}_{model_name}',
              nsmooth=1,
              and_mask=False,
              or_mask=False,
              reject_badpix=True,
              deredden=False,
              wave_range=range_arr,
              wave_mask=mask_arr,
              decompose_host=False,
              host_prior=False,
              host_prior_scale=0.2,
              host_line_mask=True,
              decomp_na_mask=True,
              qso_type='CZBIN1',
              npca_qso=10,
              host_type='PCA',
              npca_gal=5,
              Fe_uv_op=False,
              poly=False,
              BC=False,
              initial_guess=None,
              rej_abs_conti=True,
              n_pix_min_conti=100,
              linefit=True,
              rej_abs_line=True,
              MC=False,
              MCMC=RUN_MCMC,
              nsamp=NSAMP,
              param_file_name='qsopar.fits',
              nburn=NBURN,
              nthin=NTHIN,
              epsilon_jitter=0.,
              save_result=False,
              plot_fig=False,
              save_fig=False,
              plot_corner=False,
              verbose=False,
              kwargs_plot={'broad_fwhm': BROAD_FWHM_THRESHOLD},
              kwargs_conti_emcee={},
              kwargs_line_emcee={})
        fit_seconds = float(timeit.default_timer() - start)

    br = component_props(q, 'CIV_br', 'broad', ln_sigma_br=NARROW_CIV_MAXSIG)
    na = component_props(q, 'CIV_na', 'narrow', ln_sigma_br=NARROW_CIV_MAXSIG)
    lam_blue, lam_red, lam_half = broad_half_wavelengths(q)
    blueshift = blueshift_calc(lam_half)
    asymmetry = asym_calc(lam_blue, lam_red, br['peak'])
    asymmetry_inputs = broad_asymmetry_input_wavelengths(q)
    profile_mcmc = broad_profile_mcmc_results(q)

    row = {
        'Target': target,
        'Model': model_name,
        'file_name': target_row['file_name'],
        'fits_path': str(fits_path),
        'z_o3': target_row['z_o3_numeric'],
        'Wu_z': target_row['wu_z_numeric'],
        'fit_z': z,
        'fit_z_source': target_row['fit_z_source'],
        'range_arr': as_serializable(range_arr),
        'mask_arr': as_serializable(mask_arr),
        'br_components_forced': model_cfg['br_component'],
        'na_components_forced': model_cfg['na_component'],
        'narrow_fwhm_limit_kms': NARROW_FWHM_LIMIT_KMS,
        'lam_blue': lam_blue,
        'lam_red': lam_red,
        'lam_half': lam_half,
        'br_peak': br['peak'],
        'br_peak_err_mcmc': br['peak_err_mcmc'],
        'CIV_blueshift_kms': blueshift,
        'CIV_blueshift_err_mcmc': profile_mcmc['CIV_blueshift_err_mcmc'],
        'CIV_asym': asymmetry,
        'CIV_asym_err_mcmc': profile_mcmc['CIV_asym_err_mcmc'],
        'lam_blue_err_mcmc': profile_mcmc['lam_blue_err_mcmc'],
        'lam_red_err_mcmc': profile_mcmc['lam_red_err_mcmc'],
        'lam_half_err_mcmc': profile_mcmc['lam_half_err_mcmc'],
        'CIV_lam_p05_A': asymmetry_inputs['lam_p05'],
        'CIV_lam_p05_err_mcmc_A': profile_mcmc['lam_p05_err_mcmc'],
        'CIV_lam_p50_A': asymmetry_inputs['lam_p50'],
        'CIV_lam_p50_err_mcmc_A': profile_mcmc['lam_p50_err_mcmc'],
        'CIV_lam_p95_A': asymmetry_inputs['lam_p95'],
        'CIV_lam_p95_err_mcmc_A': profile_mcmc['lam_p95_err_mcmc'],
        'CIV_lam_blue_15pct_A': asymmetry_inputs['lam_blue_15pct'],
        'CIV_lam_blue_15pct_err_mcmc_A': profile_mcmc['lam_blue_15pct_err_mcmc'],
        'CIV_lam_red_15pct_A': asymmetry_inputs['lam_red_15pct'],
        'CIV_lam_red_15pct_err_mcmc_A': profile_mcmc['lam_red_15pct_err_mcmc'],
        'CIV_lam_blue_25pct_A': asymmetry_inputs['lam_blue_25pct'],
        'CIV_lam_blue_25pct_err_mcmc_A': profile_mcmc['lam_blue_25pct_err_mcmc'],
        'CIV_lam_red_25pct_A': asymmetry_inputs['lam_red_25pct'],
        'CIV_lam_red_25pct_err_mcmc_A': profile_mcmc['lam_red_25pct_err_mcmc'],
        'CIV_lam_blue_80pct_A': asymmetry_inputs['lam_blue_80pct'],
        'CIV_lam_blue_80pct_err_mcmc_A': profile_mcmc['lam_blue_80pct_err_mcmc'],
        'CIV_lam_red_80pct_A': asymmetry_inputs['lam_red_80pct'],
        'CIV_lam_red_80pct_err_mcmc_A': profile_mcmc['lam_red_80pct_err_mcmc'],
        'CIV_lam_peak_A': asymmetry_inputs['lam_peak'],
        'CIV_lam_peak_err_mcmc_A': profile_mcmc['lam_peak_err_mcmc'],
        'mcmc_samples_returned': profile_mcmc['mcmc_samples_returned'],
        'mcmc_valid_blueshift_samples': profile_mcmc['mcmc_valid_blueshift_samples'],
        'mcmc_valid_asym_samples': profile_mcmc['mcmc_valid_asym_samples'],
        'reduced Chi squared': get_line_value(q, '1_line_red_chi2'),
        'BIC_mcmc': get_line_value(q, '1_line_bic'),
        'FWHM_CIV': br['FWHM'],
        'FWHM_err_mcmc': br['FWHM_err_mcmc'],
        'br_EW': br['EW'],
        'br_EW_err_mcmc': br['EW_err_mcmc'],
        'br_area': br['area'],
        'br_area_err_mcmc': br['area_err_mcmc'],
        'br_sigma': br['sigma'],
        'br_sigma_err_mcmc': br['sigma_err_mcmc'],
        'br_snr': br['snr'],
        'na_FWHM': na['FWHM'],
        'na_FWHM_err_mcmc': na['FWHM_err_mcmc'],
        'na_EW': na['EW'],
        'na_EW_err_mcmc': na['EW_err_mcmc'],
        'na_area': na['area'],
        'na_area_err_mcmc': na['area_err_mcmc'],
        'na_sigma': na['sigma'],
        'na_sigma_err_mcmc': na['sigma_err_mcmc'],
        'na_peak': na['peak'],
        'na_peak_err_mcmc': na['peak_err_mcmc'],
        'na_snr': na['snr'],
        'line_min_chi2': get_line_value(q, '1_line_min_chi2'),
        'line_status': get_line_value(q, '1_line_status'),
        'line_ndof': get_line_value(q, '1_ndof'),
        'line_niter': get_line_value(q, '1_niter'),
        'snr_masked_pixels': meta['snr_masked_pixels'],
        'total_pixels': meta['total_pixels'],
        'snr_masked_fraction': meta['snr_masked_fraction'],
        'fit_seconds': fit_seconds,
        'status': 'ok',
        'error': '',
    }
    return row


In [7]:
results = []
expected = len(fit_targets) * len(MODEL_GRID)
print(f'Running {expected} fits: {len(fit_targets)} targets x {len(MODEL_GRID)} models.')

for _, target_row in fit_targets.iterrows():
    for model_name, model_cfg in MODEL_GRID.items():
        label = f"{target_row['target_name']} model {model_name}"
        print(f'Starting {label}')
        try:
            row = fit_target_model(target_row, model_name, model_cfg)
            print(
                f"Finished {label}: BIC={row['BIC_mcmc']:.3g}, "
                f"FWHM={row['FWHM_CIV']:.3g} km/s, "
                f"blueshift_err={row['CIV_blueshift_err_mcmc']:.3g} km/s, "
                f"asym_err={row['CIV_asym_err_mcmc']:.3g}"
            )
        except Exception as exc:
            row = {
                'Target': target_row['target_name'],
                'Model': model_name,
                'file_name': target_row['file_name'],
                'fits_path': str(target_row['fits_path']),
                'z_o3': target_row['z_o3_numeric'],
                'Wu_z': target_row['wu_z_numeric'],
                'fit_z': target_row['fit_z'],
                'fit_z_source': target_row['fit_z_source'],
                'range_arr': as_serializable(target_row['range_arr']),
                'mask_arr': as_serializable(target_row['mask_arr']),
                'br_components_forced': model_cfg['br_component'],
                'na_components_forced': model_cfg['na_component'],
                'narrow_fwhm_limit_kms': NARROW_FWHM_LIMIT_KMS,
                'status': 'failed',
                'error': repr(exc),
            }
            print(f'FAILED {label}: {exc!r}')
        results.append(row)
        pd.DataFrame(results).to_csv(RESULTS_CSV, index=False)

results_df = pd.DataFrame(results)
results_df.to_csv(RESULTS_CSV, index=False)
print(f'Wrote {len(results_df)} rows to {RESULTS_CSV}')
display(results_df)


Running 56 fits: 14 targets x 4 models.
Starting J1245+4348 model A


100%|██████████| 200/200 [00:01<00:00, 143.51it/s]


The chain is shorter than 50 times the integrated autocorrelation time for 2 parameter(s). Use this estimate with caution and run a longer chain!
N/50 = 4;
tau: [18.78080072 18.71265261]


100%|██████████| 200/200 [00:01<00:00, 145.05it/s]


The chain is shorter than 50 times the integrated autocorrelation time for 9 parameter(s). Use this estimate with caution and run a longer chain!
N/50 = 4;
tau: [22.63446784 22.97499858 24.56003978 21.61111217 23.61799731 24.57965158
 22.54729492 24.67805042 22.34265001]
Finished J1245+4348 model A: BIC=-217, FWHM=8.62e+03 km/s, blueshift_err=210 km/s, asym_err=0.198
Starting J1245+4348 model B


100%|██████████| 200/200 [00:01<00:00, 142.85it/s]


... (216 outputs left out)

The chain is shorter than 50 times the integrated autocorrelation time for 2 parameter(s). Use this estimate with caution and run a longer chain!
N/50 = 4;
tau: [15.859545   15.74669574]


100%|██████████| 200/200 [00:01<00:00, 119.49it/s]


The chain is shorter than 50 times the integrated autocorrelation time for 6 parameter(s). Use this estimate with caution and run a longer chain!
N/50 = 4;
tau: [16.66366915 16.17789801 16.54339019 16.20342143 17.16269348 23.19574205]
Finished J1458+3327 model D: BIC=-148, FWHM=9.25e+03 km/s, blueshift_err=196 km/s, asym_err=0.0106
Wrote 56 rows to /Users/nayera/PyQSOFit/Target_lists/civ_batch_fit_outputs_1600.csv


,Target,Model,file_name,fits_path,z_o3,Wu_z,fit_z,fit_z_source,range_arr,mask_arr,...,line_min_chi2,line_status,line_ndof,line_niter,snr_masked_pixels,total_pixels,snr_masked_fraction,fit_seconds,status,error
0,J1245+4348,A,01spec-6617-56365-0860.fits,/Users/nayera/PyQSOFit/CIV_galaxy_spectra_targ...,4.890,4.820,4.890,z (o3),"[1445.0, 1600.0]","[[1495.0, 1503.0]]",...,202.310124,1.0,390.0,774.0,2909,4595,0.633079,37.691536,ok,
1,J1245+4348,B,01spec-6617-56365-0860.fits,/Users/nayera/PyQSOFit/CIV_galaxy_spectra_targ...,4.890,4.820,4.890,z (o3),"[1445.0, 1600.0]","[[1495.0, 1503.0]]",...,203.293137,1.0,390.0,1458.0,2909,4595,0.633079,37.889436,ok,
2,J1245+4348,C,01spec-6617-56365-0860.fits,/Users/nayera/PyQSOFit/CIV_galaxy_spectra_targ...,4.890,4.820,4.890,z (o3),"[1445.0, 1600.0]","[[1495.0, 1503.0]]",...,203.258935,1.0,393.0,238.0,2909,4595,0.633079,38.133158,ok,
3,J1245+4348,D,01spec-6617-56365-0860.fits,/Users/nayera/PyQSOFit/CIV_galaxy_spectra_targ...,4.890,4.820,4.890,z (o3),"[1445.0, 1600.0]","[[1495.0, 1503.0]]",...,208.819188,1.0,393.0,93.0,2909,4595,0.633079,37.842683,ok,
4,J1100+5800,A,02spec-8184-57426-0330.fits,/Users/nayera/PyQSOFit/CIV_galaxy_spectra_targ...,4.759,4.776,4.759,z (o3),"[1445.0, 1615.0]","[[1502.0, 1513.0], [1533.0, 1537.0]]",...,224.860817,1.0,409.0,3458.0,3451,4617,0.747455,38.964945,ok,
5,J1100+5800,B,02spec-8184-57426-0330.fits,/Users/nayera/PyQSOFit/CIV_galaxy_spectra_targ...,4.759,4.776,4.759,z (o3),"[1445.0, 1615.0]","[[1502.0, 1513.0], [1533.0, 1537.0]]",...,191.161599,1.0,409.0,363.0,3451,4617,0.747455,38.180993,ok,
6,J1100+5800,C,02spec-8184-57426-0330.fits,/Users/nayera/PyQSOFit/CIV_galaxy_spectra_targ...,4.759,4.776,4.759,z (o3),"[1445.0, 1615.0]","[[1502.0, 1513.0], [1533.0, 1537.0]]",...,243.330222,1.0,411.0,313.0,3451,4617,0.747455,38.179137,ok,
7,J1100+5800,D,02spec-8184-57426-0330.fits,/Users/nayera/PyQSOFit/CIV_galaxy_spectra_targ...,4.759,4.776,4.759,z (o3),"[1445.0, 1615.0]","[[1502.0, 1513.0], [1533.0, 1537.0]]",...,198.991736,1.0,412.0,79.0,3451,4617,0.747455,38.212780,ok,
8,J1328+4445,A,03spec-6627-56369-0578.fits,/Users/nayera/PyQSOFit/CIV_galaxy_spectra_targ...,4.820,4.810,4.820,z (o3),"[1445.0, 1705.0]","[[1499.0, 1510.0], [1526.0, 1528.0], [1565.0, ...",...,406.077483,1.0,622.0,237.0,3244,4594,0.706138,38.236232,ok,
9,J1328+4445,B,03spec-6627-56369-0578.fits,/Users/nayera/PyQSOFit/CIV_galaxy_spectra_targ...,4.820,4.810,4.820,z (o3),"[1445.0, 1705.0]","[[1499.0, 1510.0], [1526.0, 1528.0], [1565.0, ...",...,404.260344,1.0,623.0,1082.0,3244,4594,0.706138,38.830427,ok,


# **STOP 2 !!!!!!!!!!!!**